# TrOCR v1: Jawi line recognition (Majalah Qalam 1950_11 + 1950_12)

Runtime: **GPU (T4)**. Every cell is safe to re-run; training resumes from the last checkpoint on Drive.

Needs on Drive in `jawi_dataset/`: `trocr_dataset.py`, `yolo_prefill.py`, the Label Studio export in `exports/`,
and the rendered pages in `dataset/<article>/pages/` (already there from task generation).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/jawi_dataset/

Mounted at /content/drive
/content/drive/MyDrive/jawi_dataset


In [ ]:
!pip install -q transformers==4.46.3 accelerate jiwer rapidfuzz sentencepiece

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import sys, json, csv, math, random, unicodedata, inspect
from pathlib import Path
for m in ['trocr_dataset', 'yolo_prefill']:
    sys.modules.pop(m, None)
sys.path.insert(0, '/content/drive/MyDrive/jawi_dataset')
import trocr_dataset as td
print(td.__file__)

/content/drive/MyDrive/jawi_dataset/trocr_dataset.py


## 1. Config (test articles must be the same ones the detector excluded)

In [ ]:
EXPORT_JSON = '/content/drive/MyDrive/jawi_dataset/exports/project-2-at-2026-10-23-20-34-05ec7cdf.json'
TEST_ARTICLES = ['1950_11_009', '1950_11_013', '1950_12_003', '1950_12_007', '1950_12_010', '1950_12_014']
OVERRIDES = {1: ('1950_11_002', 0)}
PAGES_BASE = '/content/drive/MyDrive/jawi_dataset/dataset'
DS = Path('/content/trocr_ds')                                       # local disk, fast
OUT = Path('/content/drive/MyDrive/jawi_dataset/trocr_runs/v1')      # checkpoints + results on Drive
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)

## 2. Build dataset v1.0 (crops from full-resolution pages)

In [ ]:
!pip install -q pymupdf
import fitz  # PyMuPDF

PDF_DIR = Path('/content/drive/MyDrive/Al_Qalam/Al_Qalam_Jawi_Full')
DPI = 400  # same as the original renders (~3267x4678 px)

pages, _ = td.yp.load_pages(EXPORT_JSON, OVERRIDES)
def png_path(a, i):
    return Path(PAGES_BASE) / a / 'pages' / f'page_{i:02d}.png'

missing = sorted({(p['article_id'], p['page_index']) for p in pages if not png_path(p['article_id'], p['page_index']).exists()})
print('missing pages:', missing)

for a, i in missing:
    pdf = PDF_DIR / f'{a}.pdf'
    if not pdf.exists():
        print('NO PDF for', a); continue
    doc = fitz.open(pdf)
    if i >= len(doc):
        print(f'{a}: PDF has {len(doc)} pages, annotation needs page {i}'); continue
    png_path(a, i).parent.mkdir(parents=True, exist_ok=True)
    doc[i].get_pixmap(dpi=DPI).save(str(png_path(a, i)))
    print('rendered', a, i)

still = [(a, i) for a, i in missing if not png_path(a, i).exists()]
print('still missing after render:', still)

stats = td.build_line_dataset(EXPORT_JSON, PAGES_BASE, DS, TEST_ARTICLES, overrides=OVERRIDES)
(OUT / 'dataset_stats.json').write_text(json.dumps(stats, ensure_ascii=False, indent=1), encoding='utf-8')
for k in ['lines', 'pages', 'crop_source', 'char_set_size', 'line_length_chars', 'excluded_pages', 'label_fixes']:
    print(k, stats[k])
assert stats['crop_source'].get('preview', 0) == 0, 'Some crops came from low-res previews: render the missing pages first'

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


missing pages: []
still missing after render: []
lines {'train': 3173, 'val': 366, 'test': 641}
pages 77
crop_source {'page_png': 4180}
char_set_size 75
line_length_chars {'min': 1, 'median': 41, 'p95': 49, 'max': 107}
excluded_pages [('1950_12_016', 0, 67), ('1950_12_017', 0, 77)]
label_fixes {'ڤ->ف': 1916, 'ک->ك': 465, 'ݢ->ڬ': 84, 'إ->ا': 53, 'پ->ڽ': 18}


# memory-safe Tesseract cell

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara > /dev/null
!pip install -q pytesseract jiwer
import pytesseract, jiwer, gc
from PIL import Image

def norm(t):
    return ' '.join(unicodedata.normalize('NFC', t).split())

rows = list(csv.DictReader(open(DS / 'test.csv', encoding='utf-8')))
refs = [norm(r['text']) for r in rows]
t_preds = []
for k, r in enumerate(rows):
    with Image.open(DS / 'images' / r['file_name']) as im:
        img = Image.fromarray(td.crop_for(im.convert('RGB'), r, False))
    t_preds.append(norm(pytesseract.image_to_string(img, lang='ara', config='--psm 7')))
    if k % 100 == 0: print(k, '/', len(rows)); gc.collect()

t_cer = jiwer.cer(refs, t_preds)
trocr = json.load(open(OUT / 'test_results_greedy.json'))['test_cer']
print(f'Tesseract (ara) line CER {t_cer:.4f}   vs   TrOCR v1 (greedy) {trocr:.4f}')
json.dump({'tesseract_cer': t_cer, 'trocr_cer_greedy': trocr}, open(OUT / 'baseline_tesseract.json', 'w'), indent=1)
with open(OUT / 'tesseract_predictions.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f); w.writerow(['file_name', 'reference', 'tesseract'])
    w.writerows([r['file_name'], ref, h] for r, ref, h in zip(rows, refs, t_preds))

0 / 641
100 / 641
200 / 641
300 / 641
400 / 641
500 / 641
600 / 641
Tesseract (ara) line CER 0.6431   vs   TrOCR v1 (greedy) 0.0378


## 3. Tokenizer check (paste this table into the report)

In [ ]:
from transformers import AutoTokenizer
train_texts = [r['text'] for r in csv.DictReader(open(DS / 'train.csv', encoding='utf-8'))]
results = {}
for name in ['microsoft/trocr-base-stage1', 'xlm-roberta-base']:
    tok = AutoTokenizer.from_pretrained(name)
    lens, exact, unk = [], 0, 0
    for t in train_texts:
        ids = tok(t, add_special_tokens=False).input_ids
        lens.append(len(ids))
        unk += sum(1 for i in ids if i == tok.unk_token_id)
        exact += tok.decode(ids, skip_special_tokens=True).strip() == t
    lens.sort()
    results[name] = {'round_trip_exact_%': round(100 * exact / len(train_texts), 2),
                     'unk_tokens': unk, 'tokens_per_line_mean': round(sum(lens) / len(lens), 1),
                     'tokens_p99': lens[int(0.99 * len(lens))], 'tokens_max': lens[-1]}
for k, v in results.items(): print(k, v)
(OUT / 'tokenizer_check.json').write_text(json.dumps(results, indent=1), encoding='utf-8')
# Decision rule: use the TrOCR checkpoint if its round trip is ~100% with 0 unknown tokens
# (byte-level BPE can encode any Unicode). xlm-roberta is shown for comparison only.

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


microsoft/trocr-base-stage1 {'round_trip_exact_%': 100.0, 'unk_tokens': 0, 'tokens_per_line_mean': 39.9, 'tokens_p99': 95, 'tokens_max': 120}
xlm-roberta-base {'round_trip_exact_%': 20.3, 'unk_tokens': 5618, 'tokens_per_line_mean': 16.7, 'tokens_p99': 38, 'tokens_max': 51}


312

## 4. Data pipeline with augmentation

In [ ]:
import torch
from PIL import Image, ImageFilter, ImageEnhance
import io
from transformers import TrOCRProcessor

CKPT = 'microsoft/trocr-base-stage1'
processor = TrOCRProcessor.from_pretrained(CKPT)
tok = processor.tokenizer
MAXLEN = min(512, results[CKPT]['tokens_max'] + 8)
print('MAXLEN', MAXLEN)

def augment(img):
    if random.random() < 0.5:
        img = img.rotate(random.uniform(-1.5, 1.5), resample=Image.BICUBIC, expand=True, fillcolor=(255, 255, 255))
    if random.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.0)))
    if random.random() < 0.5:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.3:
        buf = io.BytesIO(); img.save(buf, format='JPEG', quality=random.randint(40, 85)); img = Image.open(buf).convert('RGB')
    return img   # no flips: they would break right-to-left script

class LineDS(torch.utils.data.Dataset):
    def __init__(self, split, train):
        self.rows = list(csv.DictReader(open(DS / f'{split}.csv', encoding='utf-8')))
        self.train = train
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(DS / 'images' / r['file_name']).convert('RGB')
        img = Image.fromarray(td.crop_for(img, r, self.train))
        if self.train:
            img = augment(img)
        pixel_values = processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        ids = tok(r['text'], add_special_tokens=False, truncation=True, max_length=MAXLEN - 1).input_ids + [tok.eos_token_id]
        labels = ids + [-100] * (MAXLEN - len(ids))
        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

train_ds, val_ds, test_ds = LineDS('train', True), LineDS('val', False), LineDS('test', False)
print(len(train_ds), len(val_ds), len(test_ds))

MAXLEN 128
3173 366 641


# Small Norm

In [ ]:
   import jiwer, unicodedata
   from transformers import VisionEncoderDecoderModel
   def norm(t):
       return ' '.join(unicodedata.normalize('NFC', t).split())
   print((OUT / 'best').exists())   # must print True

True


## 5. Train (about 1-1.5 h on a T4; resumes automatically)

In [ ]:
from transformers import VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
import jiwer

model = VisionEncoderDecoderModel.from_pretrained(CKPT)
model.config.decoder_start_token_id = tok.bos_token_id
model.config.pad_token_id = tok.pad_token_id
model.config.eos_token_id = tok.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.generation_config.decoder_start_token_id = tok.bos_token_id
model.generation_config.pad_token_id = tok.pad_token_id
model.generation_config.eos_token_id = tok.eos_token_id
model.generation_config.max_length = MAXLEN
model.generation_config.no_repeat_ngram_size = 0   # OCR text legitimately repeats characters

def norm(t):
    return ' '.join(unicodedata.normalize('NFC', t).split())

def compute_metrics(pred):
    preds = pred.predictions
    labels = pred.label_ids.copy()
    labels[labels == -100] = tok.pad_token_id
    hyp = [norm(s) for s in tok.batch_decode(preds, skip_special_tokens=True)]
    ref = [norm(s) for s in tok.batch_decode(labels, skip_special_tokens=True)]
    return {'cer': jiwer.cer(ref, hyp)}

args = Seq2SeqTrainingArguments(
    output_dir=str(OUT / 'checkpoints'), per_device_train_batch_size=8, per_device_eval_batch_size=16,
    learning_rate=5e-5, warmup_ratio=0.05, weight_decay=0.01, num_train_epochs=25, fp16=True,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model='cer', greater_is_better=False, predict_with_generate=True,
    generation_max_length=MAXLEN, generation_num_beams=1, logging_steps=50, report_to='none',
    dataloader_num_workers=2, seed=SEED)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                         data_collator=default_data_collator, compute_metrics=compute_metrics,
                         callbacks=[EarlyStoppingCallback(early_stopping_patience=5)])
last = get_last_checkpoint(str(OUT / 'checkpoints')) if (OUT / 'checkpoints').exists() else None
print('resume from', last)
trainer.train(resume_from_checkpoint=last)
trainer.save_model(str(OUT / 'best'))
processor.save_pretrained(str(OUT / 'best'))
json.dump(trainer.state.log_history, open(OUT / 'log_history.json', 'w'), indent=1)
print('best val CER', trainer.state.best_metric)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

resume from None


Epoch,Training Loss,Validation Loss,Cer
1,2.039200,1.542772,0.678232
2,0.727200,0.578495,0.270187
3,0.482400,0.314448,0.251765
4,0.342100,0.238064,0.211237
5,0.331600,0.224205,0.115290
6,0.222200,0.235429,0.189515
7,0.208400,0.184046,0.095103
8,0.187500,0.160174,0.086429
9,0.197600,0.126697,0.068314
10,0.148100,0.138726,0.067240


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

best val CER 0.02709548664415106


## 6. Test set: line CER, character accuracy, per-glyph errors, confidence

In [ ]:
import gc
from rapidfuzz.distance import Levenshtein
from collections import Counter
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(2)
NUM_BEAMS = 4 if device == 'cuda' else 1
BS = 16 if device == 'cuda' else 4
DEC = 'beam4' if NUM_BEAMS > 1 else 'greedy'
model = VisionEncoderDecoderModel.from_pretrained(str(OUT / 'best')).to(device).eval()
model.config.decoder.use_cache = True
model.generation_config.use_cache = True

rows = test_ds.rows
PARTIAL = OUT / f'test_partial_{DEC}.jsonl'
done = sum(1 for _ in open(PARTIAL, encoding='utf-8')) if PARTIAL.exists() else 0
print('already done:', done)
for i in range(done, len(test_ds), BS):
    batch = torch.stack([test_ds[j]['pixel_values'] for j in range(i, min(i + BS, len(test_ds)))]).to(device)
    with torch.inference_mode():
        seqs = model.generate(batch, num_beams=NUM_BEAMS, max_length=MAXLEN)
    hyps = [norm(s) for s in tok.batch_decode(seqs, skip_special_tokens=True)]
    with open(PARTIAL, 'a', encoding='utf-8') as f:
        for h in hyps:
            f.write(json.dumps({'pred': h}, ensure_ascii=False) + '\n')
    del batch, seqs; gc.collect()
    if (i // BS) % 10 == 0: print(f'{min(i + BS, len(test_ds))}/{len(test_ds)}')

preds = [json.loads(l)['pred'] for l in open(PARTIAL, encoding='utf-8')]
refs = [norm(r['text']) for r in rows]
cer = jiwer.cer(refs, preds)
print(f'TEST line CER {cer:.4f}  character accuracy {1 - cer:.4f}  (decoding: {DEC})')

per_article = {}
for r, ref, hyp in zip(rows, refs, preds):
    a = per_article.setdefault(r['article_id'], [0, 0])
    a[0] += Levenshtein.distance(ref, hyp); a[1] += len(ref)
for a, (e, n) in sorted(per_article.items()): print(f'  {a}: CER {e / n:.4f} over {n} chars')

ref_count, err_count = Counter(), Counter()
for ref, hyp in zip(refs, preds):
    ref_count.update(ref)
    for op in Levenshtein.editops(ref, hyp):
        if op.tag in ('replace', 'delete'):
            err_count[ref[op.src_pos]] += 1
print('\nPer-letter error rate (Jawi-specific letters):')
for ch in 'ڠڽڬچۏةف':
    if ref_count[ch]: print(f'  {ch}: {err_count[ch] / ref_count[ch]:.3f}  ({ref_count[ch]} occurrences)')

with open(OUT / f'test_predictions_{DEC}.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f); w.writerow(['file_name', 'article_id', 'reference', 'prediction', 'edit_distance'])
    for r, ref, hyp in zip(rows, refs, preds):
        w.writerow([r['file_name'], r['article_id'], ref, hyp, Levenshtein.distance(ref, hyp)])
json.dump({'test_cer': cer, 'char_accuracy': 1 - cer, 'decoding': DEC,
           'per_article': {a: e / n for a, (e, n) in per_article.items()}, 'lines': len(rows)},
          open(OUT / f'test_results_{DEC}.json', 'w'), indent=1)

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

already done: 0
4/641
44/641
84/641
124/641
164/641
204/641
244/641
284/641
324/641
364/641
404/641
444/641
484/641
524/641
564/641
604/641
641/641
TEST line CER 0.0378  character accuracy 0.9622  (decoding: greedy)
  1950_11_009: CER 0.0654 over 5517 chars
  1950_11_013: CER 0.0773 over 957 chars
  1950_12_003: CER 0.0131 over 4895 chars
  1950_12_007: CER 0.0268 over 2802 chars
  1950_12_010: CER 0.0265 over 7762 chars
  1950_12_014: CER 0.0742 over 1320 chars

Per-letter error rate (Jawi-specific letters):
  ڠ: 0.009  (676 occurrences)
  ڽ: 0.026  (229 occurrences)
  ڬ: 0.027  (220 occurrences)
  چ: 0.023  (88 occurrences)
  ۏ: 1.000  (1 occurrences)
  ة: 0.036  (196 occurrences)
  ف: 0.010  (729 occurrences)


# Section 6 with CPU safe version

## 7. Tesseract baseline on the same test crops

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara > /dev/null
!pip install -q pytesseract
import pytesseract
t_preds = []
for r in rows:
    img = Image.fromarray(td.crop_for(Image.open(DS / 'images' / r['file_name']).convert('RGB'), r, False))
    t_preds.append(norm(pytesseract.image_to_string(img, lang='ara', config='--psm 7')))
t_cer = jiwer.cer(refs, t_preds)
print(f'Tesseract (ara) line CER {t_cer:.4f}   vs   TrOCR v1 {cer:.4f}')
json.dump({'tesseract_cer': t_cer, 'trocr_cer': cer}, open(OUT / 'baseline_tesseract.json', 'w'), indent=1)